In [ ]:
import os, json, random, hashlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

from sklearn.model_selection import train_test_split


from pathlib import Path
import shutil


import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import losses
from tensorflow.keras import callbacks
from tensorflow.keras import layers
from tensorflow.keras import optimizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Conv2D , MaxPooling2D , Dropout , BatchNormalization , Dense , Input , Flatten
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.applications import EfficientNetB0 , ResNet50 ,  MobileNetV2 , DenseNet121


SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

In [ ]:
ROOT = Path("/kaggle/input/datasets/salmansajid05/oral-diseases")

CLASS_FOLDERS = {
    "Calculus": ROOT / "Calculus/Calculus",
    "Caries": ROOT / "Data caries/Data caries/caries orignal data set/done",
    "Gingivitis": ROOT / "Gingivitis/Gingivitis",
    "Hypodontia": ROOT / "hypodontia/hypodontia",
    "Mouth Ulcer": ROOT / "Mouth Ulcer/Mouth Ulcer/ulcer original dataset/ulcer original dataset",
    "Tooth Discoloration": ROOT / "Tooth Discoloration/Tooth Discoloration /tooth discoloration original dataset/tooth discoloration original dataset",
}
EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

IMG_SIZE = (224, 224)
BATCH = 32
NUM_CLASSES = len(CLASS_FOLDERS)

WORK = Path("/kaggle/working")
MODEL_DIR = WORK / "models"

In [ ]:
def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

rows = []
for label, folder in CLASS_FOLDERS.items():
    if not Path(folder).exists():
        raise FileNotFoundError(folder)
    for p in sorted(Path(folder).rglob("*")):
        if p.is_file() and p.suffix.lower() in EXTS:
            rows.append((str(p), label))          

df = pd.DataFrame(rows, columns=["path", "label"])
print("raw images:", len(df))
print(df.label.value_counts(), "\n")

# 1) exact duplicates (same bytes)
df["md5"] = df.path.map(md5)
n_lab = df.groupby("md5").label.nunique()
conflict = n_lab[n_lab > 1].index
print("identical images that appear under more than one class:", len(conflict))
df = df[~df.md5.isin(conflict)]              

before = len(df)
df = df.drop_duplicates("md5").reset_index(drop=True)
print("exact duplicates removed:", before - len(df))

In [ ]:
def dhash_bits(img, size=8):
    g = img.convert("L").resize((size + 1, size), Image.LANCZOS)
    a = np.asarray(g, dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).ravel().astype(np.float32)

H, HF, keep = [], [], []
for i, p in enumerate(df.path):
    try:
        with Image.open(p) as im:
            im = im.convert("RGB")
            H.append(dhash_bits(im))
            HF.append(dhash_bits(ImageOps.mirror(im)))
        keep.append(i)
    except Exception:
        print("unreadable, dropped:", p)
df = df.iloc[keep].reset_index(drop=True)
H, HF = np.stack(H), np.stack(HF)

def hamming(A, B):                   
    return A @ (1 - B).T + (1 - A) @ B.T

NEAR_DUP_THRESHOLD = 4                 
parent = np.arange(len(df))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for s in range(0, len(df), 512):
    e = min(s + 512, len(df))
    d = np.minimum(hamming(H[s:e], H), hamming(H[s:e], HF))
    ii, jj = np.where(d <= NEAR_DUP_THRESHOLD)
    for a, b in zip(ii + s, jj):
        if a < b:
            union(a, b)

df["group"] = [find(i) for i in range(len(df))]
print("images:", len(df), "| independent groups:", df.group.nunique())


g_lab = df.groupby("group").label.nunique()
bad = g_lab[g_lab > 1].index
print("groups spanning several classes:", len(bad), "->", int(df.group.isin(bad).sum()), "images dropped")
df = df[~df.group.isin(bad)].reset_index(drop=True)
print(df.label.value_counts())

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
folds = [idx for _, idx in sgkf.split(df, df.label, groups=df.group)]

df["split"] = "train"
df.loc[folds[0], "split"] = "test"
df.loc[folds[1], "split"] = "val"
assert df.groupby("group").split.nunique().max() == 1, "leak between splits!"

print(pd.crosstab(df.label, df.split, margins=True))

class_names = sorted(df.label.unique())
cls2id = {c: i for i, c in enumerate(class_names)}
assert len(class_names) == NUM_CLASSES

train_df, val_df, test_df = [df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test")]

counts = train_df.label.value_counts().reindex(class_names).values.astype("float32")
CW_POWER = 0.5
cw = (counts.max() / counts) ** CW_POWER
cw = cw / (cw * counts).sum() * counts.sum()
print({c: round(float(w), 3) for c, w in zip(class_names, cw)})

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
CLASS_W = tf.constant(cw, dtype=tf.float32)

def _load(path, y):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE, antialias=True)
    return tf.cast(tf.round(img), tf.uint8), y           

def make_ds(d, training=False):
    y = d.label.map(cls2id).values.astype("int32")
    ds = tf.data.Dataset.from_tensor_slices((d.path.values, y))
    ds = ds.map(_load, num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(d), seed=SEED, reshuffle_each_iteration=True)

    def finish(img, yy):
        img = tf.cast(img, tf.float32)                  
        onehot = tf.one_hot(yy, NUM_CLASSES)
        if training:
            return img, onehot, tf.gather(CLASS_W, yy)   
        return img, onehot

    return ds.map(finish, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE), y

train_ds, y_train = make_ds(train_df, training=True)
val_ds, y_val = make_ds(val_df)
test_ds, y_test = make_ds(test_df)

x, y, w = next(iter(train_ds))
plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(x[i].numpy().astype("uint8"))
    plt.title(class_names[int(tf.argmax(y[i]))]); plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def make_augment():
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomTranslation(0.08, 0.08),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.2),
        layers.RandomBrightness(0.15, value_range=(0, 255)),
    ], name="augment")

@keras.utils.register_keras_serializable(package="oral")
class BackbonePreprocess(layers.Layer):
    """Backbone-specific preprocessing, saved inside the model (input = raw RGB 0..255)."""
    def __init__(self, kind="none", **kwargs):
        super().__init__(**kwargs)
        self.kind = kind
    def call(self, x):
        x = tf.cast(x, tf.float32)
        if self.kind == "resnet50":
            return keras.applications.resnet50.preprocess_input(x)
        if self.kind == "densenet121":
            return keras.applications.densenet.preprocess_input(x)
        if self.kind == "mobilenetv2":
            return keras.applications.mobilenet_v2.preprocess_input(x)
        return x                                          
    def get_config(self):
        cfg = super().get_config()
        cfg["kind"] = self.kind
        return cfg

class MacroF1(keras.callbacks.Callback):
    """Adds val_macro_f1 to the logs so callbacks can monitor it (must come first in the list)."""
    def __init__(self, ds, y_true):
        super().__init__()
        self.ds, self.y_true = ds, y_true
    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(self.ds, verbose=0).argmax(1)
        logs["val_macro_f1"] = f1_score(self.y_true, pred, average="macro")

MONITOR, MODE = "val_loss", "min"     

def get_callbacks(model_name, patience=7):
    """Your original three callbacks: best-model checkpoint, early stopping, lr reduction."""
    checkpoint = keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / f"{model_name}_best.keras"),
        monitor=MONITOR, save_best_only=True, save_weights_only=False, mode=MODE, verbose=1)
    early_stop = keras.callbacks.EarlyStopping(
        monitor=MONITOR, patience=patience, restore_best_weights=True, mode=MODE, verbose=1)
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor=MONITOR, factor=0.2, patience=3, min_lr=1e-7, mode=MODE, verbose=1)

    return [MacroF1(val_ds, y_val), checkpoint, early_stop, reduce_lr]

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
                  metrics=["accuracy"])

def predict_probs(model, ds, tta=False):
    p = model.predict(ds, verbose=0)
    if tta:                                               
        flipped = ds.map(lambda x, y: (tf.image.flip_left_right(x), y))
        p = (p + model.predict(flipped, verbose=0)) / 2
    return p

def evaluate(model, ds, y_true, title, tta=False, plot=True):
    pred = predict_probs(model, ds, tta).argmax(1)
    out = {"accuracy": accuracy_score(y_true, pred), "macro_f1": f1_score(y_true, pred, average="macro")}
    if plot:
        print(title)
        print(classification_report(y_true, pred, target_names=class_names, digits=4))
        cm = confusion_matrix(y_true, pred)
        fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax[0])
        sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt=".2f", cmap="Blues",
                    xticklabels=class_names, yticklabels=class_names, ax=ax[1])
        ax[0].set_title(title + " (counts)"); ax[1].set_title("row-normalised (recall per class)")
        for a in ax:
            a.set_xlabel("Predicted"); a.set_ylabel("True"); a.tick_params(axis="x", rotation=45)
        plt.tight_layout(); plt.show()
    return out

def plot_history(hist, title):
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    for a, (k, t) in zip(ax, [("loss", "Loss"), ("accuracy", "Accuracy"), ("macro_f1", "Macro-F1")]):
        a.plot(hist[k], label="train") if k in hist else None
        a.plot(hist["val_" + k], label="val"); a.set_title(f"{title} - {t}"); a.legend(); a.grid(True)
    plt.tight_layout(); plt.show()

def merge_hist(*hs):
    out = {}
    for h in hs:
        for k, v in h.items():
            out.setdefault(k, []).extend(v)
    return out

results, trained = {}, {}      

In [ ]:
def conv_block(x, f, drop):
    for _ in range(2):
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    return layers.Dropout(drop)(x)

def build_custom_cnn():
    inputs = keras.Input((*IMG_SIZE, 3))
    x = make_augment()(inputs)
    x = layers.Rescaling(1 / 255.0)(x)
    for f, d in [(32, 0.1), (64, 0.1), (128, 0.2), (256, 0.2), (256, 0.3)]:
        x = conv_block(x, f, d)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="CustomCNN")

cnn = build_custom_cnn()
compile_model(cnn, 1e-3)
h = cnn.fit(train_ds, validation_data=val_ds, epochs=60, callbacks=get_callbacks("CustomCNN", patience=10), verbose=2)
plot_history(h.history, "CustomCNN")
trained["CustomCNN"] = cnn
results["CustomCNN"] = evaluate(cnn, val_ds, y_val, "CustomCNN - validation")

In [ ]:
BACKBONES = {
    "EfficientNetB0": (keras.applications.EfficientNetB0, "efficientnet"),
    "ResNet50":       (keras.applications.ResNet50, "resnet50"),
    "DenseNet121":    (keras.applications.DenseNet121, "densenet121"),
    "MobileNetV2":    (keras.applications.MobileNetV2, "mobilenetv2"),
}

def build_tl_model(name, dropout=0.3):
    ctor, kind = BACKBONES[name]
    backbone = ctor(include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
    backbone.trainable = False
    inputs = keras.Input((*IMG_SIZE, 3))
    x = make_augment()(inputs)
    x = BackbonePreprocess(kind)(x)
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name=name), backbone

def unfreeze_top(backbone, frac=0.30):
    backbone.trainable = True
    cut = int(len(backbone.layers) * (1 - frac))
    for i, l in enumerate(backbone.layers):
        l.trainable = (i >= cut) and not isinstance(l, layers.BatchNormalization)

def train_backbone(name, head_epochs=10, ft_epochs=30, ft_lr=2e-5):
    print("=" * 70, f"\n{name}\n" + "=" * 70)
    model, backbone = build_tl_model(name)

    compile_model(model, 1e-3)                                          # stage 1: head only
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=head_epochs,
                   callbacks=get_callbacks(name + "_stage1", patience=5), verbose=1)

    unfreeze_top(backbone, 0.30)                                        # stage 2: fine-tune
    compile_model(model, ft_lr)
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=ft_epochs,
                   callbacks=get_callbacks(name, patience=8), verbose=1)
    return model, merge_hist(h1.history, h2.history)

MODELS_TO_TRAIN = ["EfficientNetB0", "ResNet50", "DenseNet121", "MobileNetV2"]        

for name in MODELS_TO_TRAIN:
    model, hist = train_backbone(name)
    plot_history(hist, name)
    trained[name] = model
    results[name] = evaluate(model, val_ds, y_val, f"{name} - validation")

In [ ]:
val_table = pd.DataFrame(results).T.sort_values("macro_f1", ascending=False)
print("VALIDATION"); display(val_table.round(4))

best_name = val_table.index[0]
print("best model on validation:", best_name)

test_table = pd.DataFrame({n: evaluate(m, test_ds, y_test, n, plot=False) for n, m in trained.items()}).T
print("TEST (reported for all models, selection was done on validation)"); display(test_table.loc[val_table.index].round(4))

In [ ]:
best_model = trained[best_name]
print("Test, no TTA:", evaluate(best_model, test_ds, y_test, best_name, plot=False))
print("Test, with flip-TTA:", evaluate(best_model, test_ds, y_test, best_name, tta=True, plot=False))

_ = evaluate(best_model, test_ds, y_test, f"{best_name} - TEST", tta=True)

In [ ]:
best_model.save(MODEL_DIR / "oral_best_model.keras")
with open(MODEL_DIR / "class_names.json", "w") as f:
    json.dump(class_names, f)
print("saved:", os.listdir(MODEL_DIR))